# Qwen-2.5-3B: Zero-Gated BlockAttnRes & AttnRes-GOR-v2 (Logic + Code + Tool + OOD)

Thí nghiệm kiểm chứng trực tiếp kiến trúc **Dense thắng cuộc từ Alba MiniLLM** khi ghép lên **mô hình pretrain thật `Qwen/Qwen2.5-3B` (3.09B, 36 tầng)** bằng cơ chế **Zero-Gated Residual Patching** (`alpha = 0, beta = 0` tại step 0 để bảo toàn 100% tri thức pretrain của Qwen).

Đối chứng 4 nhánh công bằng trên cùng tập train/eval **Logic (`GSM8K + ARC`) + Coding (`MBPP + Python-Alpaca`) + Tool Calling (`xlam-60k + Multi-Turn Recovery`) + External OOD (`MMLU + HumanEval + WikiText-2`)**:
1. `Qwen-2.5-3B (Base Untouched)`: Mốc gốc chưa fine-tune
2. `Qwen-2.5-3B (Baseline SFT)`: Pre-Norm chuẩn + `AdamW`
3. `Qwen-2.5-3B + Zero-Gated BlockAttnRes`: Ghép depth-attention `BlockAttnRes` (`N=9` blocks, `gate=tanh(alpha)` init `0`)
4. `Qwen-2.5-3B + Zero-Gated AttnRes-GOR-v2`: Ghép `BlockAttnRes + GOR-v2` trên Attention (`alpha=0, beta=0` init)

In [ ]:
import os
import torch

os.system("nvidia-smi")
os.system("pip install -q transformers accelerate datasets matplotlib torch")
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
    print("BF16 Supported:", torch.cuda.is_bf16_supported())

In [ ]:
import os
import subprocess
import sys

repo_dir = "/content/Alba-model"
if os.path.exists(repo_dir):
    subprocess.run(["rm", "-rf", repo_dir], check=True)
subprocess.run(
    ["git", "clone", "-b", "Khang", "https://github.com/potatips/Alba-model.git", repo_dir],
    check=True,
)
work_dir = os.path.join(repo_dir, "qwen_upcycling")
if work_dir not in sys.path:
    sys.path.insert(0, work_dir)
%cd /content/Alba-model/qwen_upcycling
os.system("python3 -m unittest test_qwen_graft -v")

In [ ]:
%cd /content/Alba-model/qwen_upcycling
!python3 qwen_train_upcycled.py \
    --mode residual_graft \
    --model-name Qwen/Qwen2.5-3B \
    --steps 240 \
    --seq-len 512 \
    --lr 2.5e-5 \
    --graft-lr 5e-3 \
    --output-dir /content/qwen_graft_results

In [ ]:
import json
from IPython.display import Image, display

res_dir = "/content/qwen_graft_results"
with open(f"{res_dir}/qwen3b_residual_graft_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

print("=== QWEN-2.5-3B RESIDUAL GRAFT BENCHMARK ===")
for name, v in results.items():
    print(
        f"{name:38s} | Overall={v['overall_bpb']:.4f} | Logic={v['logic_bpb']:.4f} | "
        f"Code={v['code_bpb']:.4f} | Tool={v.get('tool_bpb', 0.0):.4f} | OOD={v.get('ood_bpb', 0.0):.4f} | "
        f"Gates={v.get('mean_abs_alpha_attn', 0.0)}/{v.get('mean_abs_beta_gor', 0.0)} | {v['throughput_tok_s']:,.0f} tok/s"
    )

display(Image(filename=f"{res_dir}/qwen3b_residual_graft_grid.png"))

In [ ]:
from google.colab import files

res_dir = "/content/qwen_graft_results"
files.download(f"{res_dir}/qwen3b_residual_graft_results.json")
files.download(f"{res_dir}/qwen3b_residual_graft_grid.png")
files.download(f"{res_dir}/qwen3b_residual_graft_report.html")